# Query Performance Visualization
Comparison of query runtimes across different dataset sizes (100%, 50%, 25%)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

# ── Data ──────────────────────────────────────────────────────────────────────
queries = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7-a', 'Q7-b', 'Q8']

runtime_100 = [5.502, 35,    4.178, 49,    18,    15,    180,   14,    695]
runtime_50  = [2.220, 30,    3.674, 17,    11,    8.238, 25,    8.822, 352]
runtime_25  = [1.216, 18,    1.793, 9.626, 7.222, 6.345, 20,    6.637, 168]

x = np.arange(len(queries))
width = 0.26          # bar width

# ── Style ─────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

COLORS = {
    '100%': '#3A86FF',
    '50%':  '#FF6B6B',
    '25%':  '#2EC4B6',
}

## Chart 1 — Grouped Bar Chart (all queries)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

bars1 = ax.bar(x - width, runtime_100, width, label='100% dataset', color=COLORS['100%'], alpha=0.92, edgecolor='white', linewidth=0.6)
bars2 = ax.bar(x,          runtime_50,  width, label='50% dataset',  color=COLORS['50%'],  alpha=0.92, edgecolor='white', linewidth=0.6)
bars3 = ax.bar(x + width,  runtime_25,  width, label='25% dataset',  color=COLORS['25%'],  alpha=0.92, edgecolor='white', linewidth=0.6)

# Annotate bars that are large enough to label
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        if h >= 10:          # only label bars ≥ 10 s to keep it readable
            ax.text(
                bar.get_x() + bar.get_width() / 2, h + 4,
                f'{h:g}', ha='center', va='bottom', fontsize=7.5, color='#333'
            )

ax.set_xticks(x)
ax.set_xticklabels(queries, fontsize=11)
ax.set_ylabel('Runtime (s)', fontsize=12)
ax.set_title('Query Runtime by Dataset Size', fontsize=15, fontweight='bold', pad=14)
ax.legend(fontsize=11, framealpha=0.3)
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.grid(axis='y', linestyle='--', alpha=0.45, which='major')
ax.grid(axis='y', linestyle=':', alpha=0.2,  which='minor')

plt.tight_layout()
plt.savefig('query_runtime_grouped_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → query_runtime_grouped_bar.png')

## Chart 2 — Log-scale Grouped Bar (highlights large variance)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

ax.bar(x - width, runtime_100, width, label='100% dataset', color=COLORS['100%'], alpha=0.92, edgecolor='white', linewidth=0.6)
ax.bar(x,          runtime_50,  width, label='50% dataset',  color=COLORS['50%'],  alpha=0.92, edgecolor='white', linewidth=0.6)
ax.bar(x + width,  runtime_25,  width, label='25% dataset',  color=COLORS['25%'],  alpha=0.92, edgecolor='white', linewidth=0.6)

ax.set_yscale('log')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f'{y:g}'))
ax.set_xticks(x)
ax.set_xticklabels(queries, fontsize=11)
ax.set_ylabel('Runtime (s)  [log scale]', fontsize=12)
ax.set_title('Query Runtime by Dataset Size  (Log Scale)', fontsize=15, fontweight='bold', pad=14)
ax.legend(fontsize=11, framealpha=0.3)
ax.grid(axis='y', linestyle='--', alpha=0.4, which='both')

plt.tight_layout()
plt.savefig('query_runtime_log_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → query_runtime_log_bar.png')

## Chart 3 — Line Chart (trend across dataset sizes)

In [ ]:
sizes = [100, 50, 25]   # x-axis

# Each query as a list: [runtime_100, runtime_50, runtime_25]
data = {
    q: [r100, r50, r25]
    for q, r100, r50, r25 in zip(queries, runtime_100, runtime_50, runtime_25)
}

# Use a qualitative color palette (tab10)
palette = plt.cm.tab10.colors

fig, ax = plt.subplots(figsize=(10, 6))

for i, (q, runtimes) in enumerate(data.items()):
    ax.plot(sizes, runtimes, marker='o', linewidth=2.2, markersize=7,
            color=palette[i % len(palette)], label=q)
    # Annotate last point
    ax.annotate(f'{runtimes[-1]:g}s',
                xy=(25, runtimes[-1]),
                xytext=(5, 2), textcoords='offset points',
                fontsize=8, color=palette[i % len(palette)])

ax.set_xticks(sizes)
ax.set_xticklabels(['100%', '50%', '25%'], fontsize=12)
ax.set_xlabel('Dataset Size', fontsize=12)
ax.set_ylabel('Runtime (s)', fontsize=12)
ax.set_title('Runtime Trend as Dataset Size Decreases', fontsize=15, fontweight='bold', pad=14)
ax.legend(fontsize=9, ncol=2, framealpha=0.3, loc='upper right')
ax.grid(linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('query_runtime_line.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → query_runtime_line.png')

## Chart 4 — Speedup Ratio (100% as baseline)

In [ ]:
speedup_50 = [r100 / r50  for r100, r50  in zip(runtime_100, runtime_50)]
speedup_25 = [r100 / r25  for r100, r25  in zip(runtime_100, runtime_25)]

fig, ax = plt.subplots(figsize=(13, 5))

half_w = 0.3
bars_50 = ax.bar(x - half_w/2, speedup_50, half_w, label='50% vs 100%', color=COLORS['50%'],  alpha=0.88, edgecolor='white')
bars_25 = ax.bar(x + half_w/2, speedup_25, half_w, label='25% vs 100%', color=COLORS['25%'],  alpha=0.88, edgecolor='white')

# Annotate
for bars in [bars_50, bars_25]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.05,
                f'{h:.2f}×', ha='center', va='bottom', fontsize=8, color='#333')

ax.axhline(1, color='gray', linestyle='--', linewidth=1.2, label='baseline (1×)')
ax.set_xticks(x)
ax.set_xticklabels(queries, fontsize=11)
ax.set_ylabel('Speedup Factor (×)', fontsize=12)
ax.set_title('Speedup Ratio Relative to 100% Dataset', fontsize=15, fontweight='bold', pad=14)
ax.legend(fontsize=11, framealpha=0.3)
ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('query_speedup_ratio.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → query_speedup_ratio.png')